In [0]:
from pyspark.sql import functions as F

# Padronização dos nomes dos schemas e volumes
CATALOG = "vcredit_medallion"
BRONZE_SCHEMA_NAME = "bronze"
SILVER_SCHEMA_NAME = "silver"
GOLD_SCHEMA_NAME = "gold"

BRONZE_SCHEMA = f"{CATALOG}.{BRONZE_SCHEMA_NAME}"
SILVER_SCHEMA = f"{CATALOG}.{SILVER_SCHEMA_NAME}"
GOLD_SCHEMA = f"{CATALOG}.{GOLD_SCHEMA_NAME}"

LANDING_PATH = f"/Volumes/{CATALOG}/{BRONZE_SCHEMA_NAME}/landing"

# Símbolos padronizados de log
OK = "✅"
FAIL = "❌"
WARN = "⚠️"

# Para verificação visual
print(f"CATALOG: {CATALOG}")
print(f"BRONZE_SCHEMA: {BRONZE_SCHEMA}")
print(f"SILVER_SCHEMA: {SILVER_SCHEMA}")
print(f"GOLD_SCHEMA: {GOLD_SCHEMA}")
print(f"LANDING_PATH: {LANDING_PATH}")


CATALOG: vcredit_medallion
BRONZE_SCHEMA: vcredit_medallion.bronze
SILVER_SCHEMA: vcredit_medallion.silver
GOLD_SCHEMA: vcredit_medallion.gold
LANDING_PATH: /Volumes/vcredit_medallion/bronze/landing


In [0]:
# =============================================================================
# Constantes adicionais
# =============================================================================

# Horários (usado em converter_horarios)
CAMPOS_HORARIO = [
    "hora_abertura_chamado",
    "hora_inicio_atendimento",
    "hora_finalizacao_atendimento",
]

# Período do case (usado em sinalizar_periodo_horarios)
DATA_INICIO_CASE = "2025-01-01 00:00:00"
DATA_FIM_CASE_EXCLUSIVA = "2026-08-01 00:00:00"

# Colunas de comparação na deduplicação
COLUNAS_CUSTOS = ["id_custo", "id_chamado", "custo"]
COLUNAS_PESQUISAS = ["id_pesquisa", "id_chamado", "nota_atendimento"]
COLUNAS_CHAMADOS_HORA = [
    "id_chamado", "id_cliente",
    "hora_abertura_chamado",
    "hora_inicio_atendimento",
    "hora_finalizacao_atendimento",
]

# Domínios e regex usados na célula de clientes
DOMINIOS_EMAIL = [
    "gmail.com", "hotmail.com", "outlook.com", "globo.com", "terra.com.br",
    "uol.com.br", "bol.com.br", "ig.com.br", "yahoo.com.br",
]
REGEX_EMAIL = r"^[a-z0-9._%+\-]+@[a-z0-9\-]+(\.[a-z]+)+$"
IDADE_MIN, IDADE_MAX = 18, 100
DIA_VENC_MIN, DIA_VENC_MAX = 1, 31

print(f"{OK} constantes adicionais carregadas")

✅ constantes adicionais carregadas


In [0]:
# =============================================================================
# HELPERS CONSOLIDADOS
# =============================================================================

# ---------------------- Leitura / limpeza / tipagem ----------------------
def limpar_texto(coluna):
    """Trim + colapsa espaços internos + vazio -> null."""
    c = F.trim(F.regexp_replace(coluna, r"\s+", " "))
    return F.when(c == "", F.lit(None)).otherwise(c)


def para_inteiro(coluna):
    """Converte texto numérico em int; não numérico vira null."""
    c = limpar_texto(coluna)
    return F.when(c.rlike(r"^\d+$"), c.try_cast("int"))


def para_data(coluna):
    """Aceita dd/MM/yyyy e yyyy-MM-dd; outros formatos viram null."""
    c = F.trim(coluna)
    return (
        F.when(c.rlike(r"^\d{2}/\d{2}/\d{4}$"), F.to_date(c, "dd/MM/yyyy"))
         .when(c.rlike(r"^\d{4}-\d{2}-\d{2}$"), F.to_date(c, "yyyy-MM-dd"))
    )


# ---------------------- Validação / logging ----------------------
def verificar(descricao, qtd, critico=False):
    """Loga OK/WARN/FAIL. Se critico e qtd>0, interrompe o notebook."""
    if qtd == 0:
        print(f"{OK} {descricao}: 0")
    elif critico:
        print(f"{FAIL} {descricao}: {qtd}")
        raise Exception(f"Validação crítica falhou: {descricao} ({qtd})")
    else:
        print(f"{WARN} {descricao}: {qtd}")


def relatorio_invalidados(df_bronze, df_silver, colunas):
    """Conta valores que existiam na Bronze e viraram null na Silver."""
    b = df_bronze.agg(*[F.count(limpar_texto(F.col(c))).alias(c) for c in colunas]).first()
    s = df_silver.agg(*[F.count(F.col(c)).alias(c) for c in colunas]).first()
    for c in colunas:
        verificar(f"{c}: valores preenchidos invalidados", b[c] - s[c])


# ---------------------- Gravação ----------------------
def gravar_silver(df, nome_tabela):
    """Grava DataFrame na Silver como Delta (overwrite + overwriteSchema)."""
    tabela = f"{SILVER_SCHEMA}.{nome_tabela}"
    (df.write.format("delta")
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(tabela))
    print(f"{OK} {tabela}: {spark.table(tabela).count()} linhas")


def gravar_gold(df, nome_tabela):
    """Grava DataFrame na Gold como Delta, adicionando _gold_atualizado_em."""
    tabela = f"{GOLD_SCHEMA}.{nome_tabela}"
    (df.withColumn("_gold_atualizado_em", F.current_timestamp())
       .write.format("delta")
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(tabela))
    print(f"{OK} {tabela}: {spark.table(tabela).count()} linhas")


# ---------------------- Chave canônica ----------------------
def chave(col):
    """Normaliza texto para comparação."""
    texto = F.lower(F.trim(col))
    texto = F.translate(texto, "áàâãäéèêëíìîïóòôõöúùûüç", "aaaaaeeeeiiiiooooouuuuc")
    texto = F.regexp_replace(texto, "[^a-z0-9]", "")
    texto = F.regexp_replace(texto, "atendimento", "atend")
    return texto


# ---------------------- Identificadores ----------------------
def identificador_ausente(nome_coluna):
    """True se o id for nulo, vazio ou texto 'null'."""
    coluna = F.col(nome_coluna)
    texto = F.lower(F.trim(coluna.try_cast("string")))
    return coluna.isNull() | texto.isin("", "null")


# ---------------------- Conversões ----------------------
def converter_custos(dados):
    """Converte texto de custo em decimal(18,2), preservando original."""
    return (
        dados.withColumnRenamed("custo", "custo_original")
        .withColumn("custo_texto", F.lower(F.trim(F.col("custo_original"))))
        .withColumn("custo_texto", F.regexp_replace(F.col("custo_texto"), r"^r\$\s*", ""))
        .withColumn("custo_texto", F.regexp_replace(F.col("custo_texto"), r"\s*reais$", ""))
        .withColumn("custo_texto", F.regexp_replace(F.trim(F.col("custo_texto")), ",", "."))
        .withColumn("custo", F.when(
            F.col("custo_texto").rlike(r"^[+-]?[0-9]+(?:\.[0-9]{1,2})?$"),
            F.expr("try_cast(custo_texto as decimal(18,2))")))
        .withColumn("custo_nao_convertido", F.col("custo").isNull())
        .withColumn("custo_negativo", F.coalesce(F.col("custo") < 0, F.lit(False)))
        .drop("custo_texto")
    )


def converter_pesquisas(dados):
    """Classifica notas em válida/ausente/fora_dominio e cria nota int."""
    return (
        dados.withColumnRenamed("nota_atendimento", "nota_atendimento_original")
        .withColumn("nota_texto", F.trim(F.col("nota_atendimento_original")))
        .withColumn("nota_ausente",
            F.col("nota_atendimento_original").isNull()
            | F.lower(F.col("nota_texto")).isin("", "null"))
        .withColumn("nota_valida",
            F.coalesce(F.col("nota_texto").rlike(r"^[1-5]$"), F.lit(False)))
        .withColumn("nota_fora_dominio",
            ~F.col("nota_ausente") & ~F.col("nota_valida"))
        .withColumn("nota_atendimento",
            F.when(F.col("nota_valida"), F.expr("try_cast(nota_texto as int)")))
        .drop("nota_texto")
    )


def converter_horarios(dados):
    """Converte 'DD/MM/AAAA às HH:MM:SS' em timestamp_ntz; preserva original."""
    resultado = dados
    padrao = r"^(\d{2})/(\d{2})/(\d{4})\s+às\s+(\d{2}:\d{2}:\d{2})$"
    for campo in CAMPOS_HORARIO:
        campo_original = f"{campo}_original"
        resultado = resultado.withColumnRenamed(campo, campo_original)
        texto = F.trim(F.col(campo_original))
        resultado = (
            resultado
            .withColumn("horario_texto",
                F.when(texto.rlike(padrao),
                       F.regexp_replace(texto, padrao, "$3-$2-$1 $4")))
            .withColumn(campo, F.expr("try_cast(horario_texto as timestamp_ntz)"))
            .drop("horario_texto")
        )
    return resultado


# ---------------------- Cálculos temporais ----------------------
def calcular_tempos_horarios(dados):
    """Calcula tempos em segundos, sinaliza ordem temporal inválida."""
    resultado = dados
    intervalos = [
        ("espera",      "hora_abertura_chamado",   "hora_inicio_atendimento"),
        ("atendimento", "hora_inicio_atendimento", "hora_finalizacao_atendimento"),
        ("total",       "hora_abertura_chamado",   "hora_finalizacao_atendimento"),
    ]
    for nome, ini, fim in intervalos:
        resultado = resultado.withColumn(
            f"tempo_{nome}_calculado_segundos",
            (F.col(fim) - F.col(ini)).try_cast("long"))
    resultado = (
        resultado
        .withColumn("inicio_antes_abertura",
            F.coalesce(F.col("tempo_espera_calculado_segundos") < 0, F.lit(False)))
        .withColumn("finalizacao_antes_inicio",
            F.coalesce(F.col("tempo_atendimento_calculado_segundos") < 0, F.lit(False)))
        .withColumn("finalizacao_antes_abertura",
            F.coalesce(F.col("tempo_total_calculado_segundos") < 0, F.lit(False)))
        .withColumn("ordem_temporal_invalida",
            F.col("inicio_antes_abertura")
            | F.col("finalizacao_antes_inicio")
            | F.col("finalizacao_antes_abertura"))
    )
    for nome, _, _ in intervalos:
        valor = F.col(f"tempo_{nome}_calculado_segundos")
        resultado = resultado.withColumn(
            f"tempo_{nome}_segundos",
            F.when(~F.col("ordem_temporal_invalida") & (valor >= 0), valor))
    return resultado


def sinalizar_periodo_horarios(dados):
    """Sinaliza se a abertura caiu no período do case."""
    abertura = F.col("hora_abertura_chamado")
    inicio = F.lit(DATA_INICIO_CASE).try_cast("timestamp_ntz")
    fim = F.lit(DATA_FIM_CASE_EXCLUSIVA).try_cast("timestamp_ntz")
    return (
        dados
        .withColumn("abertura_fora_periodo",
            F.coalesce((abertura < inicio) | (abertura >= fim), F.lit(False)))
        .withColumn("abertura_no_periodo",
            abertura.isNotNull() & ~F.col("abertura_fora_periodo"))
    )


# ---------------------- Cruzamentos entre fontes ----------------------
def adicionar_correspondencia_chamados(dados, referencia):
    """Left join por id_chamado, adicionando chamado_encontrado."""
    ref = referencia.select("id_chamado").distinct().withColumn("chamado_encontrado", F.lit(True))
    return dados.join(ref, on="id_chamado", how="left").fillna({"chamado_encontrado": False})


def cruzar_referencia_horarios(dados, referencia):
    """Cruza horários com chamados, comparando id_cliente."""
    ref = referencia.withColumn("chamado_encontrado", F.lit(True))
    return (
        dados
        .join(ref, on="id_chamado", how="left")
        .fillna({"chamado_encontrado": False})
        .withColumn("cliente_comparavel",
            F.col("chamado_encontrado")
            & ~identificador_ausente("id_cliente")
            & ~identificador_ausente("id_cliente_referencia"))
        .withColumn("cliente_divergente",
            F.col("cliente_comparavel")
            & ~F.col("id_cliente").eqNullSafe(F.col("id_cliente_referencia")))
    )


def comparar_tempos_informados(dados):
    """Compara tempos informados (chamados) com calculados (chamados_hora)."""
    resultado = dados
    for tipo in ["espera", "atendimento"]:
        p = f"tempo_{tipo}"
        campo_orig = f"{p}_informado_original"
        campo_info = f"{p}_informado_segundos"
        campo_calc = f"{p}_calculado_segundos"
        texto = F.trim(F.col(campo_orig))
        resultado = (
            resultado
            .withColumn(f"{p}_informado_ausente",
                texto.isNull() | F.lower(texto).isin("", "null"))
            .withColumn(campo_info,
                F.when(~F.col(f"{p}_informado_ausente") & texto.rlike(r"^[+-]?[0-9]+$"),
                       F.expr(f"try_cast(trim({campo_orig}) as bigint)")))
            .withColumn(f"{p}_informado_invalido",
                ~F.col(f"{p}_informado_ausente") & F.col(campo_info).isNull())
            .withColumn(f"{p}_informado_negativo",
                F.coalesce(F.col(campo_info) < 0, F.lit(False)))
            .withColumn(f"{p}_comparavel",
                F.col(campo_info).isNotNull() & F.col(campo_calc).isNotNull())
            .withColumn(f"{p}_divergente",
                F.when(F.col(f"{p}_comparavel"), F.col(campo_calc) != F.col(campo_info)))
            .withColumn(f"diferenca_{p}_segundos",
                F.when(F.col(f"{p}_comparavel"), F.col(campo_calc) - F.col(campo_info)))
        )
    return resultado


# ---------------------- Agregações auxiliares ----------------------
def somar_custos(dados):
    """Soma decimal de custo, retornando 0 quando tudo é null."""
    resultado = dados.agg(
        F.coalesce(F.sum("custo"), F.lit(0).try_cast("decimal(38,2)")).alias("total")
    ).first()
    return resultado["total"]


print(f"{OK} helpers carregados")

✅ helpers carregados


## tb_base_motivos

**Origem:** `bronze.tb_base_motivos`
**Destino:** `silver.tb_base_motivos`

**Transformações:**
- Trim em todas as colunas de texto (a coluna `_ingerido_em` é preservada como timestamp).
- `id_motivo`: `try_cast` para `bigint`.
- `categoria`: preenche nulos com base no dicionário do case (ids 12, 14, 16).
- `criticidade`: padronizada para `alta` / `media` / `baixa` (lowercase, cobre variante `média`).

**Sobre a ausência de `_origem_*`:** esta tabela não preserva colunas de auditoria porque `categoria` original era `null` (o preenchimento vem do dicionário) e `criticidade` sofre apenas normalização de caixa — não há ambiguidade semântica a preservar.

**Validações:**
- Nenhuma linha perdida (crítico).
- `id_motivo` sem nulos e sem duplicatas (crítico).
- `categoria` sem nulos (crítico — lista os motivos não mapeados no `display` antes de falhar).
- `criticidade` dentro do domínio — warning.
- `_ingerido_em` nulo (crítico).
- Idempotência: reexecução da transformação produz o mesmo resultado (`exceptAll` bidirecional, crítico).

In [0]:
df_bronze = spark.table(f"{BRONZE_SCHEMA}.tb_base_motivos")
linhas_bronze = df_bronze.count()


def construir_silver_base_motivos(bronze):
    return (
        bronze
        .select(*[F.trim(F.col(c)).alias(c) for c in bronze.columns if c != "_ingerido_em"]
                + [F.col("_ingerido_em")])
        .withColumn("id_motivo", F.col("id_motivo").try_cast("bigint"))
        .withColumn("categoria",
            F.when(F.col("categoria").isNull() & (F.col("id_motivo") == 14), F.lit("Cadastral"))
             .when(F.col("categoria").isNull() & (F.col("id_motivo") == 16), F.lit("Cadastral"))
             .when(F.col("categoria").isNull() & (F.col("id_motivo") == 12), F.lit("Cartao"))
             .otherwise(F.col("categoria")))
        .withColumn("criticidade",
            F.when(F.lower(F.col("criticidade")) == "alta",  F.lit("alta"))
             .when(F.lower(F.col("criticidade")) == "media", F.lit("media"))
             .when(F.lower(F.col("criticidade")) == "média", F.lit("media"))
             .when(F.lower(F.col("criticidade")) == "baixa", F.lit("baixa"))
             .otherwise(F.col("criticidade")))
    )


df_silver = construir_silver_base_motivos(df_bronze)

# ---- Validações ----
verificar("linhas perdidas (Bronze != Silver)",
          linhas_bronze - df_silver.count(), critico=True)

verificar("id_motivo nulo",
          df_silver.filter(F.col("id_motivo").isNull()).count(), critico=True)

verificar("id_motivo duplicado",
          df_silver.count() - df_silver.select("id_motivo").distinct().count(), critico=True)

verificar("nome_motivo nulo",
          df_silver.filter(F.col("nome_motivo").isNull()).count(), critico=True)

categorias_nulas = df_silver.filter(F.col("categoria").isNull())
n_categorias_nulas = categorias_nulas.count()
if n_categorias_nulas > 0:
    print(f"{FAIL} categoria nula: {n_categorias_nulas} — motivos sem mapeamento:")
    display(categorias_nulas.select("id_motivo", "nome_motivo"))
    raise Exception("Categoria nula — atualize o dicionário de mapeamento antes de publicar.")
else:
    print(f"{OK} categoria nula: 0")

verificar("criticidade fora de alta/media/baixa",
          df_silver.filter(~F.col("criticidade").isin("alta", "media", "baixa")).count())

verificar("_ingerido_em nulo",
          df_silver.filter(F.col("_ingerido_em").isNull()).count(), critico=True)

# ---- Idempotência ----
df_reexecucao = construir_silver_base_motivos(df_bronze)
verificar("reexecução: linhas ausentes",
          df_silver.exceptAll(df_reexecucao).count(), critico=True)
verificar("reexecução: linhas adicionais",
          df_reexecucao.exceptAll(df_silver).count(), critico=True)

# ---- Gravação ----
gravar_silver(df_silver, "tb_base_motivos")
display(spark.table(f"{SILVER_SCHEMA}.tb_base_motivos").orderBy("id_motivo"))

✅ linhas perdidas (Bronze != Silver): 0
✅ id_motivo nulo: 0
✅ id_motivo duplicado: 0
✅ nome_motivo nulo: 0
✅ categoria nula: 0
✅ criticidade fora de alta/media/baixa: 0
✅ _ingerido_em nulo: 0
✅ reexecução: linhas ausentes: 0
✅ reexecução: linhas adicionais: 0
✅ vcredit_medallion.silver.tb_base_motivos: 14 linhas


id_motivo,nome_motivo,categoria,criticidade,_ingerido_em
3,Contestação de fatura,Financeiro,alta,2026-10-26T20:21:12.593Z
4,"Alteração de dados cadastrais (vencimento da fatura, telefone, email)",Cadastral,baixa,2026-10-26T20:21:12.593Z
5,Consulta de limite,Financeiro,baixa,2026-10-26T20:21:12.593Z
6,Desbloqueio de cartão,Cartao,alta,2026-10-26T20:21:12.593Z
7,Compra não autorizada,Cartao,alta,2026-10-26T20:21:12.593Z
8,Consulta de contrato,Financeiro,media,2026-10-26T20:21:12.593Z
9,Problema com aplicativo,Atendimento,media,2026-10-26T20:21:12.593Z
10,Consulta de fatura,Financeiro,baixa,2026-10-26T20:21:12.593Z
11,Bloqueio de cartão,Cartao,alta,2026-10-26T20:21:12.593Z
12,Contratação de cartão adicional,Cartao,media,2026-10-26T20:21:12.593Z


## tb_canais

**Origem:** `bronze.tb_canais`
**Destino:** `silver.tb_canais`

**Transformações:**
- Trim em todas as colunas de texto (a coluna `_ingerido_em` é preservada como timestamp).
- `id_canal`: `try_cast` para `bigint`.

**Sobre a ausência de `_origem_*`:** esta tabela não preserva colunas de auditoria porque **não há transformação de valor** — apenas trim de espaços e cast do id.

**Validações:**
- Nenhuma linha perdida (crítico).
- `id_canal` sem nulos e sem duplicatas (crítico).
- `nome_canal` sem nulos (crítico — chave de negócio usada em outros notebooks).
- `tipo_canal` dentro do domínio (`Humano` / `Autosservico`) — warning.
- `status_canal` dentro do domínio (`ativo` / `inativo`) — warning.
- `_ingerido_em` nulo (crítico).
- Idempotência: reexecução da transformação produz o mesmo resultado (`exceptAll` bidirecional, crítico).

In [0]:
df_bronze = spark.table(f"{BRONZE_SCHEMA}.tb_canais")
linhas_bronze = df_bronze.count()


def construir_silver_canais(bronze):
    return (
        bronze
        .select(*[F.trim(F.col(c)).alias(c) for c in bronze.columns if c != "_ingerido_em"]
                + [F.col("_ingerido_em")])
        .withColumn("id_canal", F.col("id_canal").try_cast("bigint"))
    )


df_silver = construir_silver_canais(df_bronze)

# ---- Validações ----
verificar("linhas perdidas (Bronze != Silver)",
          linhas_bronze - df_silver.count(), critico=True)

verificar("id_canal nulo",
          df_silver.filter(F.col("id_canal").isNull()).count(), critico=True)

verificar("id_canal duplicado",
          df_silver.count() - df_silver.select("id_canal").distinct().count(), critico=True)

verificar("nome_canal nulo",
          df_silver.filter(F.col("nome_canal").isNull()).count(), critico=True)

verificar("tipo_canal fora de Humano/Autosservico",
          df_silver.filter(~F.col("tipo_canal").isin("Humano", "Autosservico")).count())

verificar("status_canal fora de ativo/inativo",
          df_silver.filter(~F.col("status_canal").isin("ativo", "inativo")).count())

verificar("_ingerido_em nulo",
          df_silver.filter(F.col("_ingerido_em").isNull()).count(), critico=True)

# ---- Idempotência ----
df_reexecucao = construir_silver_canais(df_bronze)
verificar("reexecução: linhas ausentes",
          df_silver.exceptAll(df_reexecucao).count(), critico=True)
verificar("reexecução: linhas adicionais",
          df_reexecucao.exceptAll(df_silver).count(), critico=True)

# ---- Gravação ----
gravar_silver(df_silver, "tb_canais")
display(spark.table(f"{SILVER_SCHEMA}.tb_canais").orderBy("id_canal"))


✅ linhas perdidas (Bronze != Silver): 0
✅ id_canal nulo: 0
✅ id_canal duplicado: 0
✅ nome_canal nulo: 0
✅ tipo_canal fora de Humano/Autosservico: 0
✅ status_canal fora de ativo/inativo: 0
✅ _ingerido_em nulo: 0
✅ reexecução: linhas ausentes: 0
✅ reexecução: linhas adicionais: 0
✅ vcredit_medallion.silver.tb_canais: 4 linhas


id_canal,nome_canal,tipo_canal,status_canal,_ingerido_em
1,URA,Autosservico,ativo,2026-10-26T20:21:18.644Z
2,CHATBOT,Autosservico,ativo,2026-10-26T20:21:18.644Z
3,ATENDIMENTO INICIAL,Humano,ativo,2026-10-26T20:21:18.644Z
4,ATENDIMENTO ESPECIALIZADO,Humano,ativo,2026-10-26T20:21:18.644Z


## tb_clientes

**Origem:** `bronze.tb_clientes`
**Destino:** `silver.tb_clientes`

**Transformações:**
- `id_cliente`: mantido string (preserva zeros à esquerda).
- `nome`: limpeza de espaços.
- `email`: remove espaços e lowercase; se faltar `@`, reinsere antes do domínio conhecido; valida contra `REGEX_EMAIL` (formato inválido vira null).
- `regiao`: grafia única das 5 regiões (fora dessas → null).
- `idade`: inteiro; fora da faixa 18–100 → null.
- `data_cadastro`: `dd/MM/yyyy` ou `yyyy-MM-dd` → date.
- `dia_vencimento_fatura`: inteiro; fora de 1–31 → null.
- `_ingerido_em`: preservado.

**Validações:**
- Nenhuma linha perdida (crítico).
- `id_cliente` sem nulos e sem duplicatas (crítico).
- `id_cliente` sempre com 11 dígitos (warning).
- Relatório de valores que existiam na Bronze e viraram null (warning por coluna).
- `data_cadastro` no futuro (warning).
- `_ingerido_em` nulo (crítico).
- Idempotência: reexecução da transformação produz o mesmo resultado (`exceptAll` bidirecional, crítico).

In [0]:
df_bronze = spark.table(f"{BRONZE_SCHEMA}.tb_clientes")
linhas_bronze = df_bronze.count()


def construir_silver_clientes(bronze):
    email_base = F.lower(F.regexp_replace(F.col("email"), r"\s+", ""))

    email_recuperado = F.lit(None).try_cast("string")
    for dominio in sorted(DOMINIOS_EMAIL, key=len, reverse=True):
        email_recuperado = F.coalesce(
            email_recuperado,
            F.when(
                email_base.endswith(dominio) & (F.length(email_base) > len(dominio)),
                F.concat(
                    email_base.substr(F.lit(1), F.length(email_base) - F.lit(len(dominio))),
                    F.lit("@"),
                    F.lit(dominio),
                ),
            ),
        )
    email_corrigido = F.when(email_base.contains("@"), email_base).otherwise(email_recuperado)
    email_final = F.when(email_corrigido.rlike(REGEX_EMAIL), email_corrigido)

    regiao_chave = F.lower(F.trim(F.col("regiao")))
    regiao_final = (
        F.when(regiao_chave == "norte", "Norte")
        .when(regiao_chave == "nordeste", "Nordeste")
        .when(regiao_chave == "centro-oeste", "Centro-Oeste")
        .when(regiao_chave == "sudeste", "Sudeste")
        .when(regiao_chave == "sul", "Sul")
    )

    idade_num = para_inteiro(F.col("idade"))
    dia_num = para_inteiro(F.col("dia_vencimento_fatura"))

    return bronze.select(
        limpar_texto(F.col("id_cliente")).alias("id_cliente"),
        limpar_texto(F.col("nome")).alias("nome"),
        email_final.alias("email"),
        regiao_final.alias("regiao"),
        F.when(idade_num.between(IDADE_MIN, IDADE_MAX), idade_num).alias("idade"),
        para_data(F.col("data_cadastro")).alias("data_cadastro"),
        F.when(dia_num.between(DIA_VENC_MIN, DIA_VENC_MAX), dia_num).alias("dia_vencimento_fatura"),
        F.col("_ingerido_em"),
    )


df_silver = construir_silver_clientes(df_bronze)

# ---- Validações ----
verificar("linhas perdidas (Bronze != Silver)",
          linhas_bronze - df_silver.count(), critico=True)

verificar("id_cliente nulo",
          df_silver.filter(F.col("id_cliente").isNull()).count(), critico=True)

verificar("id_cliente duplicado",
          df_silver.count() - df_silver.select("id_cliente").distinct().count(), critico=True)

verificar("id_cliente fora do formato de 11 dígitos",
          df_silver.filter(~F.col("id_cliente").rlike(r"^\d{11}$")).count())

relatorio_invalidados(df_bronze, df_silver,
                      ["nome", "email", "regiao", "idade",
                       "data_cadastro", "dia_vencimento_fatura"])

verificar("data_cadastro futura",
          df_silver.filter(F.col("data_cadastro") > F.current_date()).count())

colunas_negocio = [c for c in df_silver.columns if c not in ("id_cliente", "_ingerido_em")]
verificar("cadastros duplicados com id_cliente diferente (mantidos)",
          df_silver.count() - df_silver.dropDuplicates(colunas_negocio).count())

verificar("_ingerido_em nulo",
          df_silver.filter(F.col("_ingerido_em").isNull()).count(), critico=True)

# ---- Idempotência ----
df_reexecucao = construir_silver_clientes(df_bronze)
verificar("reexecução: linhas ausentes",
          df_silver.exceptAll(df_reexecucao).count(), critico=True)
verificar("reexecução: linhas adicionais",
          df_reexecucao.exceptAll(df_silver).count(), critico=True)

# ---- Gravação ----
gravar_silver(df_silver, "tb_clientes")
display(spark.table(f"{SILVER_SCHEMA}.tb_clientes").limit(20))

✅ linhas perdidas (Bronze != Silver): 0
✅ id_cliente nulo: 0
✅ id_cliente duplicado: 0
✅ id_cliente fora do formato de 11 dígitos: 0
✅ nome: valores preenchidos invalidados: 0
✅ email: valores preenchidos invalidados: 0
✅ regiao: valores preenchidos invalidados: 0
⚠️ idade: valores preenchidos invalidados: 380
✅ data_cadastro: valores preenchidos invalidados: 0
✅ dia_vencimento_fatura: valores preenchidos invalidados: 0
✅ data_cadastro futura: 0
⚠️ cadastros duplicados com id_cliente diferente (mantidos): 705
✅ _ingerido_em nulo: 0
✅ reexecução: linhas ausentes: 0
✅ reexecução: linhas adicionais: 0
✅ vcredit_medallion.silver.tb_clientes: 47705 linhas


id_cliente,nome,email,regiao,idade,data_cadastro,dia_vencimento_fatura,_ingerido_em
56324416706,Sérgio Martins Miranda,sergio.miranda98@hotmail.com,Sul,51,2024-10-14,10,2026-10-26T20:21:41.010Z
97653229900,Leandro Magalhães Lopes,lopes.leandro@yahoo.com.br,Sudeste,51,2026-07-07,10,2026-10-26T20:21:41.010Z
15132904850,Giovana Soares Vasconcelos,giovana.vasconcelos26@hotmail.com,Sudeste,52,2019-02-16,25,2026-10-26T20:21:41.010Z
62566924227,Adriana Nascimento,adriana96@gmail.com,Nordeste,39,2025-08-27,20,2026-10-26T20:21:41.010Z
35213062014,Vera Duarte,vera.duarte@ig.com.br,Sudeste,34,2026-05-07,15,2026-10-26T20:21:41.010Z
11173485227,Renata Freitas Fernandes,renata96@yahoo.com.br,Sudeste,37,2025-01-26,5,2026-10-26T20:21:41.010Z
50855203771,Lucas Cardoso,lucascardoso@uol.com.br,Nordeste,38,2026-07-15,28,2026-10-26T20:21:41.010Z
72534560781,Camila Guimarães Gomes,camila_gomes@uol.com.br,Nordeste,80,2025-09-03,25,2026-10-26T20:21:41.010Z
40409080977,Thiago Peixoto,thiago98@gmail.com,Sul,92,2023-06-21,25,2026-10-26T20:21:41.010Z
65936173686,Mariana Aparecida Esteves,esteves.mariana@ig.com.br,Norte,41,2025-05-31,15,2026-10-26T20:21:41.010Z


## tb_base_atendentes

**Origem:** `bronze.tb_base_atendentes`
**Destino:** `silver.tb_base_atendentes`

**Transformações:**
- `id_atendente`: inteiro via `para_inteiro`, com cast para `bigint`.
- `nome_atendente`: limpeza de espaços.
- `nivel_atendimento`: extrai o dígito de `"1"`, `"Nível 1"`, `"N1"`, `"2"`, `"Nível 2"`, `"N2"` e converte para int.
- `data_admissao`: `dd/MM/yyyy` → date.
- `data_desligamento`: `dd/MM/yyyy` → date; vazio vira null (atendente ativo).
- `status_atendente`: `lower(limpar_texto(...))` → `ativo` / `inativo`.
- `_ingerido_em`: preservado.

**Validações:**
- Nenhuma linha perdida (crítico).
- `id_atendente` sem nulos e sem duplicatas (crítico).
- Relatório de valores que existiam na Bronze e viraram null (warning por coluna).
- `status_atendente` fora de `ativo`/`inativo` (warning).
- `status_atendente` incoerente com `data_desligamento` (warning).
- `data_desligamento` anterior a `data_admissao` (warning).
- `nivel_atendimento` fora de 1 ou 2 (warning).
- `_ingerido_em` nulo (crítico).
- Idempotência: reexecução da transformação produz o mesmo resultado (`exceptAll` bidirecional, crítico).

In [0]:
df_bronze = spark.table(f"{BRONZE_SCHEMA}.tb_base_atendentes")
linhas_bronze = df_bronze.count()


def construir_silver_atendentes(bronze):
    return bronze.select(
        para_inteiro(F.col("id_atendente")).try_cast("bigint").alias("id_atendente"),
        limpar_texto(F.col("nome_atendente")).alias("nome_atendente"),
        para_inteiro(
            F.regexp_extract(F.col("nivel_atendimento"), r"(\d+)", 1)
        ).alias("nivel_atendimento"),
        para_data(F.col("data_admissao")).alias("data_admissao"),
        para_data(F.col("data_desligamento")).alias("data_desligamento"),
        F.lower(limpar_texto(F.col("status_atendente"))).alias("status_atendente"),
        F.col("_ingerido_em"),
    )


df_silver = construir_silver_atendentes(df_bronze)

# ---- Validações ----
verificar("linhas perdidas (Bronze != Silver)",
          linhas_bronze - df_silver.count(), critico=True)

verificar("id_atendente nulo",
          df_silver.filter(F.col("id_atendente").isNull()).count(), critico=True)

verificar("id_atendente duplicado",
          df_silver.count() - df_silver.select("id_atendente").distinct().count(), critico=True)

relatorio_invalidados(df_bronze, df_silver,
                      ["nome_atendente", "nivel_atendimento",
                       "data_admissao", "data_desligamento", "status_atendente"])

verificar("status_atendente fora de ativo/inativo",
          df_silver.filter(~F.col("status_atendente").isin("ativo", "inativo")).count())

verificar("status incoerente com data_desligamento",
          df_silver.filter(
              (F.col("status_atendente") == "inativo") != F.col("data_desligamento").isNotNull()
          ).count())

verificar("desligamento anterior à admissão",
          df_silver.filter(F.col("data_desligamento") < F.col("data_admissao")).count())

verificar("nivel_atendimento fora de 1 ou 2",
          df_silver.filter(~F.col("nivel_atendimento").isin(1, 2)).count())

verificar("_ingerido_em nulo",
          df_silver.filter(F.col("_ingerido_em").isNull()).count(), critico=True)

# ---- Idempotência ----
df_reexecucao = construir_silver_atendentes(df_bronze)
verificar("reexecução: linhas ausentes",
          df_silver.exceptAll(df_reexecucao).count(), critico=True)
verificar("reexecução: linhas adicionais",
          df_reexecucao.exceptAll(df_silver).count(), critico=True)

# ---- Gravação ----
gravar_silver(df_silver, "tb_base_atendentes")
display(spark.table(f"{SILVER_SCHEMA}.tb_base_atendentes").orderBy("id_atendente"))

✅ linhas perdidas (Bronze != Silver): 0
✅ id_atendente nulo: 0
✅ id_atendente duplicado: 0
✅ nome_atendente: valores preenchidos invalidados: 0
✅ nivel_atendimento: valores preenchidos invalidados: 0
✅ data_admissao: valores preenchidos invalidados: 0
✅ data_desligamento: valores preenchidos invalidados: 0
✅ status_atendente: valores preenchidos invalidados: 0
✅ status_atendente fora de ativo/inativo: 0
✅ status incoerente com data_desligamento: 0
✅ desligamento anterior à admissão: 0
✅ nivel_atendimento fora de 1 ou 2: 0
✅ _ingerido_em nulo: 0
✅ reexecução: linhas ausentes: 0
✅ reexecução: linhas adicionais: 0
✅ vcredit_medallion.silver.tb_base_atendentes: 25 linhas


id_atendente,nome_atendente,nivel_atendimento,data_admissao,data_desligamento,status_atendente,_ingerido_em
1,Marcela França,1,2019-04-18,2025-11-16,inativo,2026-10-26T20:21:05.978Z
2,Zilda Esteves,1,2023-09-12,2026-01-24,inativo,2026-10-26T20:21:05.978Z
3,William Marques,1,2019-05-28,2025-04-15,inativo,2026-10-26T20:21:05.978Z
4,Solange Silveira,1,2019-10-07,null,ativo,2026-10-26T20:21:05.978Z
5,Joaquim Barros,1,2020-12-06,2026-05-23,inativo,2026-10-26T20:21:05.978Z
6,Roberta Assunção,1,2021-10-29,null,ativo,2026-10-26T20:21:05.978Z
7,Luiza Galvão,1,2022-04-24,null,ativo,2026-10-26T20:21:05.978Z
8,Josefa Brito,1,2023-01-25,null,ativo,2026-10-26T20:21:05.978Z
9,Jaqueline Campos,1,2019-04-19,null,ativo,2026-10-26T20:21:05.978Z
10,Marcos Torres,1,2020-01-20,null,ativo,2026-10-26T20:21:05.978Z


## tb_chamados

**Origem:** `bronze.tb_chamados`
**Destino:** `silver.tb_chamados`

**Depende de:** `silver.tb_base_motivos`, `silver.tb_canais`, `silver.tb_base_atendentes`, `silver.tb_clientes`

**Transformações:**
- `dropDuplicates` considerando só colunas de negócio (a `_ingerido_em` fica de fora da comparação).
- `motivo` / `canal`: normalizados via função `chave` e cruzados com as Silver de motivos/canais. Valores crus preservados em `_origem_motivo` / `_origem_canal`.
- `resolvido`: convertido para boolean a partir de variantes (`Sim`, `S`, `1`, `Não`, `N`, `0`, `Nao resolvido`). Vazio → null.
- `id_atendente`: validado contra a Silver de atendentes. Valor cru preservado em `_origem_id_atendente`; sinalizado com `atendente_identificado`.
- `id_cliente`: validado contra a Silver de clientes. Sinalizado com `cliente_encontrado`.
- `tempo_espera` / `tempo_atendimento`: convertidos para double em `tempo_*_seg_origem`.
- `id_chamado`: `bigint`.
- `_ingerido_em`: preservado.

**Convenção `_origem_*`:** colunas com prefixo `_` sinalizam rastro de auditoria do tratamento — não usar em análises. Servem para provar o que foi normalizado e para debugar chamados com `id_motivo` / `id_canal` / `id_atendente` nulos.

**Colunas de horário:** removidas desta tabela. A fonte oficial dos horários é `tb_chamados_hora`.

**Validações:**
- Nenhuma linha perdida no join (compara pós-dedup vs Silver — as duplicatas removidas são intencionais).
- `id_chamado` sem nulos e sem duplicatas (crítico).
- Todos os chamados com `id_motivo` preenchido (crítico — lista as grafias não reconhecidas em caso de falha).
- Todos os chamados com `id_canal` preenchido (crítico — idem).
- `_ingerido_em` nulo (crítico).
- Idempotência: reexecução da transformação produz o mesmo resultado (`exceptAll` bidirecional, crítico).
- Contagem de `atendente_identificado = false` e `cliente_encontrado = false` (informativo).

In [0]:
COLUNA_CONTROLE = "_ingerido_em"

ch_bronze = spark.table(f"{BRONZE_SCHEMA}.tb_chamados")
linhas_bronze = ch_bronze.count()

# Exclui o metadado de ingestão da comparação: linhas idênticas em negócio mas
# carregadas em momentos diferentes devem ser tratadas como duplicatas
colunas_dados = [c for c in ch_bronze.columns if c != COLUNA_CONTROLE]

df_sem_duplicatas = ch_bronze.dropDuplicates(colunas_dados)
linhas_dedup = df_sem_duplicatas.count()
print(f"chamados: bronze={linhas_bronze} | sem duplicatas={linhas_dedup} | removidas={linhas_bronze - linhas_dedup}")


def construir_silver_chamados(ch_bronze_df):
    ch = ch_bronze_df.dropDuplicates(colunas_dados)

    motivos = (
        spark.table(f"{SILVER_SCHEMA}.tb_base_motivos")
        .select(
            "id_motivo",
            F.col("nome_motivo").alias("motivo"),
            "categoria", "criticidade",
            chave(F.col("nome_motivo")).alias("chave_motivo"),
        )
    )
    canais = (
        spark.table(f"{SILVER_SCHEMA}.tb_canais")
        .select(
            "id_canal",
            F.col("nome_canal").alias("canal"),
            "tipo_canal",
            chave(F.col("nome_canal")).alias("chave_canal"),
        )
    )
    atendentes_validos = (
        spark.table(f"{SILVER_SCHEMA}.tb_base_atendentes")
        .select(F.col("id_atendente").alias("id_atendente_valido")).distinct()
    )
    clientes = (
        spark.table(f"{SILVER_SCHEMA}.tb_clientes")
        .select(F.col("id_cliente").alias("id_cliente_base")).distinct()
    )

    ch = (
        ch
        .withColumn("chave_motivo", chave(F.col("motivo")))
        .withColumn("chave_canal", chave(F.col("canal")))
        .withColumnRenamed("motivo", "_origem_motivo")
        .withColumnRenamed("canal", "_origem_canal")
        .join(motivos, "chave_motivo", "left")
        .join(canais, "chave_canal", "left")
    )

    resolvido_chave = chave(F.col("resolvido"))
    ch = ch.withColumn("resolvido",
        F.when(resolvido_chave.isin("sim", "s", "1"), F.lit(True))
         .when(resolvido_chave.isin("nao", "n", "0", "naoresolvido"), F.lit(False))
         .otherwise(F.lit(None).try_cast("boolean")))

    ch = (
        ch
        .withColumn("_origem_id_atendente", F.trim(F.col("id_atendente")))
        .join(
            atendentes_validos,
            F.expr("try_cast(_origem_id_atendente as int)") == F.col("id_atendente_valido"),
            "left",
        )
        .withColumn("id_atendente", F.col("id_atendente_valido"))
        .withColumn("atendente_identificado", F.col("id_atendente").isNotNull())
    )

    ch = (
        ch
        .withColumn("tempo_espera_seg_origem", F.expr("try_cast(tempo_espera as double)"))
        .withColumn("tempo_atendimento_seg_origem", F.expr("try_cast(tempo_atendimento as double)"))
        .withColumn("id_cliente", F.trim(F.col("id_cliente")))
        .join(clientes, F.col("id_cliente") == F.col("id_cliente_base"), "left")
        .withColumn("cliente_encontrado", F.col("id_cliente_base").isNotNull())
    )

    return ch.select(
        F.expr("try_cast(id_chamado as bigint)").alias("id_chamado"),
        "id_cliente", "cliente_encontrado",
        "id_motivo", "motivo", "categoria", "criticidade", "_origem_motivo",
        "id_canal", "canal", "tipo_canal", "_origem_canal",
        "resolvido",
        "id_atendente", "atendente_identificado", "_origem_id_atendente",
        "tempo_espera_seg_origem", "tempo_atendimento_seg_origem",
        F.col(COLUNA_CONTROLE),
    )


df_silver = construir_silver_chamados(ch_bronze)

# Todos os joins da construção são do tipo left; nenhuma linha é perdida entre a
# deduplicação e o resultado final
verificar("linhas perdidas no join (pós-dedup != Silver)",
          linhas_dedup - df_silver.count(), critico=True)

verificar("id_chamado nulo",
          df_silver.filter(F.col("id_chamado").isNull()).count(), critico=True)

verificar("id_chamado duplicado",
          df_silver.count() - df_silver.select("id_chamado").distinct().count(), critico=True)

sem_motivo = df_silver.filter(F.col("id_motivo").isNull())
n_sem_motivo = sem_motivo.count()
if n_sem_motivo > 0:
    print(f"{FAIL} chamados sem motivo padronizado: {n_sem_motivo} — grafias não reconhecidas:")
    display(sem_motivo.select("_origem_motivo").distinct())
    raise Exception("Motivo não reconhecido — ajuste a função `chave` ou o catálogo de motivos.")
else:
    print(f"{OK} chamados sem motivo padronizado: 0")

sem_canal = df_silver.filter(F.col("id_canal").isNull())
n_sem_canal = sem_canal.count()
if n_sem_canal > 0:
    print(f"{FAIL} chamados sem canal padronizado: {n_sem_canal} — grafias não reconhecidas:")
    display(sem_canal.select("_origem_canal").distinct())
    raise Exception("Canal não reconhecido — ajuste a função `chave` ou o catálogo de canais.")
else:
    print(f"{OK} chamados sem canal padronizado: 0")

verificar("_ingerido_em nulo",
          df_silver.filter(F.col(COLUNA_CONTROLE).isNull()).count(), critico=True)

verificar("chamados com atendente não identificado",
          df_silver.filter(~F.col("atendente_identificado")).count())
verificar("chamados com cliente não encontrado",
          df_silver.filter(~F.col("cliente_encontrado")).count())

# ---- Idempotência ----
df_reexecucao = construir_silver_chamados(ch_bronze)

if df_silver.schema != df_reexecucao.schema:
    raise Exception("Reexecução alterou o schema.")

verificar("reexecução: linhas ausentes",
          df_silver.exceptAll(df_reexecucao).count(), critico=True)
verificar("reexecução: linhas adicionais",
          df_reexecucao.exceptAll(df_silver).count(), critico=True)

# ---- Gravação ----
gravar_silver(df_silver, "tb_chamados")
display(spark.table(f"{SILVER_SCHEMA}.tb_chamados").limit(20))

chamados: bronze=134474 | sem duplicatas=131838 | removidas=2636
✅ linhas perdidas no join (pós-dedup != Silver): 0
✅ id_chamado nulo: 0
✅ id_chamado duplicado: 0
✅ chamados sem motivo padronizado: 0
✅ chamados sem canal padronizado: 0
✅ _ingerido_em nulo: 0
⚠️ chamados com atendente não identificado: 764
⚠️ chamados com cliente não encontrado: 1330
✅ reexecução: linhas ausentes: 0
✅ reexecução: linhas adicionais: 0
✅ vcredit_medallion.silver.tb_chamados: 131838 linhas


id_chamado,id_cliente,cliente_encontrado,id_motivo,motivo,categoria,criticidade,_origem_motivo,id_canal,canal,tipo_canal,_origem_canal,resolvido,id_atendente,atendente_identificado,_origem_id_atendente,tempo_espera_seg_origem,tempo_atendimento_seg_origem,_ingerido_em
52060,36276971799,true,15,Renegociação de dívida,Financeiro,alta,Renegociação de dívida,1,URA,Autosservico,Ura,false,9,true,9,null,216.0,2026-10-26T20:21:24.799Z
89333,52402176464,true,15,Renegociação de dívida,Financeiro,alta,RENEGOCIAÇÃO DE DÍVIDA,2,CHATBOT,Autosservico,CHATBOT,false,3,true,3,null,104.0,2026-10-26T20:21:24.799Z
128467,95609443006,true,9,Problema com aplicativo,Atendimento,media,Problema com aplicativo,1,URA,Autosservico,URA,false,18,true,18,null,71.0,2026-10-26T20:21:24.799Z
45000,21748174497,true,15,Renegociação de dívida,Financeiro,alta,RENEGOCIAÇÃO DE DÍVIDA,2,CHATBOT,Autosservico,CHATBOT,null,20,true,20,null,106.0,2026-10-26T20:21:24.799Z
97042,70649266905,true,16,Cancelamento de conta,Cadastral,alta,Cancelamento de conta,1,URA,Autosservico,URA,false,1,true,1,null,218.0,2026-10-26T20:21:24.799Z
118384,98181934335,true,3,Contestação de fatura,Financeiro,alta,Contestação de fatura,3,ATENDIMENTO INICIAL,Humano,Atendimento Inicial,false,13,true,13,1.0,1.0,2026-10-26T20:21:24.799Z
44717,50297589152,true,10,Consulta de fatura,Financeiro,baixa,Consulta de fatura,1,URA,Autosservico,URA,true,14,true,14,null,271.0,2026-10-26T20:21:24.799Z
91912,58605012202,true,4,"Alteração de dados cadastrais (vencimento da fatura, telefone, email)",Cadastral,baixa,"Alteração de dados cadastrais (vencimento da fatura, telefone, email)",2,CHATBOT,Autosservico,Chatbot,true,11,true,11,null,50.0,2026-10-26T20:21:24.799Z
84270,39808757879,true,9,Problema com aplicativo,Atendimento,media,Problema com aplicativo,4,ATENDIMENTO ESPECIALIZADO,Humano,Atendimento_Especializado,true,19,true,19,564.0,3.0,2026-10-26T20:21:24.799Z
128920,45998304098,true,11,Bloqueio de cartão,Cartao,alta,Bloqueio de cartao,2,CHATBOT,Autosservico,chatbot,true,19,true,19,null,52.0,2026-10-26T20:21:24.799Z


## tb_custos

**Origem:** `bronze.tb_custos`
**Destino:** `silver.tb_custos`

**Depende de:** `silver.tb_chamados`

**Transformações:**
- `id_custo` / `id_chamado`: `try_cast` para `bigint`.
- `dropDuplicates` por `(id_custo, id_chamado, custo)` — repetições exatas.
- `custo`: conversão para `decimal(18,2)`, aceitando formatos heterogêneos. Valor cru preservado em `_origem_custo`; sinalizações em `custo_nao_convertido` e `custo_negativo`.
- Cruzamento com `tb_chamados` via `chamado_encontrado` (left join).
- `_ingerido_em`: preservado.

**Convenção `_origem_*`:** rastro de auditoria do tratamento. Não usar em análises. As flags `custo_nao_convertido` / `custo_negativo` **não** têm prefixo `_` porque são para **usar** em análise.

**Validações:**
- Nenhuma linha perdida no join.
- `id_custo` com registros conflitantes após dedup (crítico).
- Nenhum custo não convertido (crítico).
- Reconciliação financeira: `soma(origem) == soma(dedup) + valor_das_repetições` (crítico).
- Idempotência: reexecução da transformação produz o mesmo resultado (`exceptAll` bidirecional, crítico).
- `id_custo` sem nulos e sem duplicatas (crítico).
- `id_chamado` sem nulos (crítico).
- `_ingerido_em` nulo (crítico).
- `chamado_encontrado = false` (informativo).

In [0]:
df_bronze = (
    spark.table(f"{BRONZE_SCHEMA}.tb_custos")
    .select(
        F.expr("try_cast(id_custo as bigint)").alias("id_custo"),
        F.expr("try_cast(id_chamado as bigint)").alias("id_chamado"),
        "custo",
        F.col("_ingerido_em"),
    )
)
linhas_bronze = df_bronze.count()


def construir_silver_custos(bronze, chamados_ref):
    sem_dup = bronze.dropDuplicates(COLUNAS_CUSTOS)
    tratados = (
        converter_custos(sem_dup)
        .withColumnRenamed("custo_original", "_origem_custo")
    )
    return adicionar_correspondencia_chamados(tratados, chamados_ref)


chamados_ref = spark.table(f"{SILVER_SCHEMA}.tb_chamados").select("id_chamado")

df_sem_duplicatas = df_bronze.dropDuplicates(COLUNAS_CUSTOS)
linhas_dedup = df_sem_duplicatas.count()
print(f"custos: bronze={linhas_bronze} | sem duplicatas={linhas_dedup} | removidas={linhas_bronze - linhas_dedup}")

conflitos = (
    df_sem_duplicatas.groupBy("id_custo").count()
    .filter(F.col("count") > 1).count()
)
verificar("id_custo com registros conflitantes após dedup", conflitos, critico=True)

df_com_correspondencia = construir_silver_custos(df_bronze, chamados_ref)

verificar("linhas perdidas no join de chamados",
          linhas_dedup - df_com_correspondencia.count(), critico=True)

# Reconciliação financeira: soma da origem deve bater com a soma da silver
# mais o valor das linhas removidas por deduplicação
df_custos_origem_conv = converter_custos(df_bronze)

nao_convertidos = df_custos_origem_conv.filter(F.col("custo_nao_convertido")).limit(1).count()
if nao_convertidos > 0:
    print(f"{FAIL} custos não convertidos na origem:")
    display(df_custos_origem_conv.filter(F.col("custo_nao_convertido")).select("custo_original").distinct())
    raise Exception("Custos não convertidos — investigue antes de publicar.")

df_grupos_repetidos = (
    df_bronze.groupBy(*COLUNAS_CUSTOS)
    .agg(F.count("*").alias("ocorrencias"))
    .filter(F.col("ocorrencias") > 1)
)
df_valores_repetidos = (
    converter_custos(df_grupos_repetidos)
    .select((F.col("custo") * (F.col("ocorrencias") - 1)).alias("custo"))
)

total_origem = somar_custos(df_custos_origem_conv)
total_sem_dup = somar_custos(df_com_correspondencia)
total_repetido = somar_custos(df_valores_repetidos)
diferenca = total_origem - total_sem_dup - total_repetido

print(f"Reconciliação: origem={total_origem} | dedup={total_sem_dup} | repetidos={total_repetido} | diferença={diferenca}")
verificar("reconciliação financeira (origem - dedup - repetidos)",
          int(diferenca != 0), critico=True)

colunas_silver = [
    "id_custo", "id_chamado", "custo", "_origem_custo",
    "custo_nao_convertido", "custo_negativo", "chamado_encontrado",
    "_ingerido_em",
]
df_silver = df_com_correspondencia.select(*colunas_silver)

verificar("id_custo nulo",
          df_silver.filter(F.col("id_custo").isNull()).count(), critico=True)

verificar("id_custo duplicado",
          df_silver.count() - df_silver.select("id_custo").distinct().count(), critico=True)

verificar("id_chamado nulo",
          df_silver.filter(F.col("id_chamado").isNull()).count(), critico=True)

verificar("custo não convertido",
          df_silver.filter(F.col("custo_nao_convertido")).count(), critico=True)

verificar("_ingerido_em nulo",
          df_silver.filter(F.col("_ingerido_em").isNull()).count(), critico=True)

verificar("custos sem chamado correspondente",
          df_silver.filter(~F.col("chamado_encontrado")).count())

# ---- Idempotência ----
df_reexecucao = construir_silver_custos(df_bronze, chamados_ref).select(*colunas_silver)

if df_silver.schema != df_reexecucao.schema:
    raise Exception("Reexecução alterou o schema.")

verificar("reexecução: linhas ausentes",
          df_silver.exceptAll(df_reexecucao).count(), critico=True)
verificar("reexecução: linhas adicionais",
          df_reexecucao.exceptAll(df_silver).count(), critico=True)

# ---- Gravação ----
gravar_silver(df_silver, "tb_custos")
display(spark.table(f"{SILVER_SCHEMA}.tb_custos").limit(20))

custos: bronze=131069 | sem duplicatas=130288 | removidas=781
✅ id_custo com registros conflitantes após dedup: 0
✅ linhas perdidas no join de chamados: 0
Reconciliação: origem=1455269.97 | dedup=1447671.52 | repetidos=7598.45 | diferença=0.00
✅ reconciliação financeira (origem - dedup - repetidos): 0
✅ id_custo nulo: 0
✅ id_custo duplicado: 0
✅ id_chamado nulo: 0
✅ custo não convertido: 0
✅ _ingerido_em nulo: 0
✅ custos sem chamado correspondente: 0
✅ reexecução: linhas ausentes: 0
✅ reexecução: linhas adicionais: 0
✅ vcredit_medallion.silver.tb_custos: 130288 linhas


id_custo,id_chamado,custo,_origem_custo,custo_nao_convertido,custo_negativo,chamado_encontrado,_ingerido_em
39350,34788,0.52,0.52,false,false,true,2026-10-26T20:21:47.387Z
84453,80433,0.99,"0,99",false,false,true,2026-10-26T20:21:47.387Z
23353,18592,17.35,17.35,false,false,true,2026-10-26T20:21:47.387Z
122622,119036,15.85,"15,85",false,false,true,2026-10-26T20:21:47.387Z
12972,8061,14.97,14.97,false,false,true,2026-10-26T20:21:47.387Z
22413,17638,0.54,"0,54",false,false,true,2026-10-26T20:21:47.387Z
115953,112299,6.78,"6,78",false,false,true,2026-10-26T20:21:47.387Z
68903,64670,39.68,"39,68",false,false,true,2026-10-26T20:21:47.387Z
97394,93513,12.02,12.02,false,false,true,2026-10-26T20:21:47.387Z
17890,13045,1.44,1.44,false,false,true,2026-10-26T20:21:47.387Z


## tb_pesquisa_satisfacao

**Origem:** `bronze.tb_pesquisa_satisfacao`
**Destino:** `silver.tb_pesquisa_satisfacao`

**Depende de:** `silver.tb_chamados`

**Transformações:**
- `id_pesquisa` / `id_chamado`: `try_cast` para `bigint`.
- `dropDuplicates` por `(id_pesquisa, id_chamado, nota_atendimento)` — repetições exatas.
- Conflito: mesmo `id_pesquisa` com `nota_atendimento` diferente (crítico).
- `nota_atendimento`: valor cru preservado em `_origem_nota_atendimento`; três flags mutuamente exclusivas (`nota_valida` 1–5, `nota_ausente` nulo/vazio/`"null"`, `nota_fora_dominio`).
- `nota_atendimento`: inteiro, preenchido apenas quando `nota_valida = true`.
- Cruzamento com `tb_chamados` via `chamado_encontrado` (left join).
- `_ingerido_em`: preservado.

**Convenção `_origem_*`:** rastro de auditoria. As flags `nota_*` não têm prefixo porque são para usar em análise.

**Granularidade:** uma linha por `id_pesquisa`. Um chamado pode ter múltiplas pesquisas.

**Validações:**
- Nenhuma linha perdida no join.
- `id_pesquisa` com registros conflitantes após dedup (crítico).
- Idempotência: `exceptAll` bidirecional (crítico).
- Reconciliação da classificação: `total == nota_valida + nota_ausente + nota_fora_dominio` (crítico).
- `notas_numericas == nota_valida` (crítico).
- `id_pesquisa` sem nulos e sem duplicatas (crítico).
- `id_chamado` sem nulos (crítico).
- `_ingerido_em` nulo (crítico).
- `chamado_encontrado = false` (informativo).
- Chamados com múltiplas pesquisas (informativo).

In [0]:
df_bronze = (
    spark.table(f"{BRONZE_SCHEMA}.tb_pesquisa_satisfacao")
    .select(
        F.expr("try_cast(id_pesquisa as bigint)").alias("id_pesquisa"),
        F.expr("try_cast(id_chamado as bigint)").alias("id_chamado"),
        "nota_atendimento",
        F.col("_ingerido_em"),
    )
)
linhas_bronze = df_bronze.count()


def construir_silver_pesquisas(bronze, chamados_ref):
    tratados = (
        converter_pesquisas(bronze.dropDuplicates(COLUNAS_PESQUISAS))
        .withColumnRenamed("nota_atendimento_original", "_origem_nota_atendimento")
    )
    return adicionar_correspondencia_chamados(tratados, chamados_ref)


chamados_ref = spark.table(f"{SILVER_SCHEMA}.tb_chamados").select("id_chamado")

df_sem_duplicatas = df_bronze.dropDuplicates(COLUNAS_PESQUISAS)
linhas_dedup = df_sem_duplicatas.count()
print(f"pesquisas: bronze={linhas_bronze} | sem duplicatas={linhas_dedup} | removidas={linhas_bronze - linhas_dedup}")

conflitos = (
    df_sem_duplicatas.groupBy("id_pesquisa").count()
    .filter(F.col("count") > 1).count()
)
verificar("id_pesquisa com registros conflitantes após dedup", conflitos, critico=True)

multiplas_pesquisas = (
    df_sem_duplicatas.groupBy("id_chamado").count()
    .filter(F.col("count") > 1).count()
)
verificar("chamados com múltiplas pesquisas (mantidos)", multiplas_pesquisas)

df_com_correspondencia = construir_silver_pesquisas(df_bronze, chamados_ref)

verificar("linhas perdidas no join de chamados",
          linhas_dedup - df_com_correspondencia.count(), critico=True)

metricas = df_com_correspondencia.agg(
    F.count("*").alias("registros"),
    F.sum(F.col("nota_valida").cast("long")).alias("nota_valida"),
    F.sum(F.col("nota_ausente").cast("long")).alias("nota_ausente"),
    F.sum(F.col("nota_fora_dominio").cast("long")).alias("nota_fora_dominio"),
    F.count("nota_atendimento").alias("notas_numericas"),
).first()

print(f"Classificação: total={metricas['registros']} | válidas={metricas['nota_valida']} | "
      f"ausentes={metricas['nota_ausente']} | fora_dominio={metricas['nota_fora_dominio']}")

verificar("total != (válidas + ausentes + fora_dominio)",
          metricas["registros"] - (metricas["nota_valida"] + metricas["nota_ausente"] + metricas["nota_fora_dominio"]),
          critico=True)

verificar("notas_numericas != nota_valida",
          metricas["notas_numericas"] - metricas["nota_valida"],
          critico=True)

colunas_silver = [
    "id_pesquisa", "id_chamado", "nota_atendimento",
    "_origem_nota_atendimento", "nota_valida", "nota_ausente", "nota_fora_dominio",
    "chamado_encontrado", "_ingerido_em",
]
df_silver = df_com_correspondencia.select(*colunas_silver)

verificar("id_pesquisa nulo",
          df_silver.filter(F.col("id_pesquisa").isNull()).count(), critico=True)

verificar("id_pesquisa duplicado",
          df_silver.count() - df_silver.select("id_pesquisa").distinct().count(), critico=True)

verificar("id_chamado nulo",
          df_silver.filter(F.col("id_chamado").isNull()).count(), critico=True)

verificar("_ingerido_em nulo",
          df_silver.filter(F.col("_ingerido_em").isNull()).count(), critico=True)

verificar("pesquisas sem chamado correspondente",
          df_silver.filter(~F.col("chamado_encontrado")).count())

# ---- Idempotência ----
df_reexecucao = construir_silver_pesquisas(df_bronze, chamados_ref).select(*colunas_silver)

if df_silver.schema != df_reexecucao.schema:
    raise Exception("Reexecução alterou o schema.")

verificar("reexecução: linhas ausentes",
          df_silver.exceptAll(df_reexecucao).count(), critico=True)
verificar("reexecução: linhas adicionais",
          df_reexecucao.exceptAll(df_silver).count(), critico=True)

# ---- Gravação ----
gravar_silver(df_silver, "tb_pesquisa_satisfacao")
display(spark.table(f"{SILVER_SCHEMA}.tb_pesquisa_satisfacao").limit(20))

pesquisas: bronze=81021 | sem duplicatas=81021 | removidas=0
✅ id_pesquisa com registros conflitantes após dedup: 0
⚠️ chamados com múltiplas pesquisas (mantidos): 802
✅ linhas perdidas no join de chamados: 0
Classificação: total=81021 | válidas=74122 | ausentes=5678 | fora_dominio=1221
✅ total != (válidas + ausentes + fora_dominio): 0
✅ notas_numericas != nota_valida: 0
✅ id_pesquisa nulo: 0
✅ id_pesquisa duplicado: 0
✅ id_chamado nulo: 0
✅ _ingerido_em nulo: 0
✅ pesquisas sem chamado correspondente: 0
✅ reexecução: linhas ausentes: 0
✅ reexecução: linhas adicionais: 0
✅ vcredit_medallion.silver.tb_pesquisa_satisfacao: 81021 linhas


id_pesquisa,id_chamado,nota_atendimento,_origem_nota_atendimento,nota_valida,nota_ausente,nota_fora_dominio,chamado_encontrado,_ingerido_em
27051,39785,3,3,true,false,false,true,2026-10-26T20:21:54.169Z
65478,104417,2,2,true,false,false,true,2026-10-26T20:21:54.169Z
1776,2517,5,5,true,false,false,true,2026-10-26T20:21:54.169Z
10373,14714,4,4,true,false,false,true,2026-10-26T20:21:54.169Z
48094,73998,5,5,true,false,false,true,2026-10-26T20:21:54.169Z
1877,2654,4,4,true,false,false,true,2026-10-26T20:21:54.169Z
26524,38977,3,3,true,false,false,true,2026-10-26T20:21:54.169Z
31000,46055,3,3,true,false,false,true,2026-10-26T20:21:54.169Z
46994,72154,1,1,true,false,false,true,2026-10-26T20:21:54.169Z
80210,131819,5,5,true,false,false,true,2026-10-26T20:21:54.169Z


## tb_chamados_hora

**Origem:** `bronze.tb_chamados_hora`
**Destino:** `silver.tb_chamados_hora`

**Depende de:** `bronze.tb_chamados` (para a referência de tempos informados)

**Transformações:**
- `id_chamado`: `try_cast` para `bigint`. `id_cliente`: mantido string (CPF).
- `dropDuplicates` por `(id_chamado, id_cliente, hora_abertura_chamado, hora_inicio_atendimento, hora_finalizacao_atendimento)`.
- Conflito: mesmo `id_chamado` com horários diferentes (crítico).
- `hora_*`: parseadas de `"DD/MM/AAAA às HH:MM:SS"` para `timestamp_ntz`. Valores crus preservados em `_origem_hora_*`.
- `tempo_*_calculado_segundos`: calculados a partir dos horários. Sinalizados os casos de ordem temporal inválida.
- `tempo_*_segundos`: preenchidos apenas quando o intervalo é válido.
- `abertura_no_periodo` / `abertura_fora_periodo`: sinalização do período do case.
- Cruzamento com `bronze.tb_chamados` para comparar `id_cliente` e tempos informados.
- `tempo_*_informado_segundos`: tempos informados pela fonte (`chamados`), preservados crus em `_origem_tempo_*_informado`. Sinaliza ausente/inválido/negativo; calcula diferença vs. o calculado.
- `_ingerido_em`: preservado.

**Convenção `_origem_*`:** rastro de auditoria.

**Nota sobre `tempo_*_segundos` vs `tempo_*_calculado_segundos`:** o sufixo `_calculado_segundos` é **auditoria** (mantém o valor mesmo quando negativo); o `tempo_*_segundos` é o **valor tratado** (nulo quando há contradição temporal). Para análise, usar sempre `tempo_*_segundos`.

**Granularidade:** uma linha por `id_chamado`.

**Validações:**
- Nenhuma linha perdida nos joins.
- `id_chamado` com registros conflitantes após dedup (crítico).
- `id_chamado` com registros conflitantes na referência de chamados (crítico).
- `id_chamado` sem nulos e sem duplicatas (crítico).
- `_ingerido_em` nulo (crítico).
- Reconciliação do período (crítico).
- Coerência dos tempos: `espera + atendimento == total` quando ambos disponíveis (crítico).
- Idempotência: `exceptAll` bidirecional (crítico).
- `id_cliente` divergente entre as duas fontes (informativo).
- Chamados sem horário na referência e horários sem chamado na referência (informativo).

In [0]:
RENAMES_ORIGEM = {
    "hora_abertura_chamado_original": "_origem_hora_abertura_chamado",
    "hora_inicio_atendimento_original": "_origem_hora_inicio_atendimento",
    "hora_finalizacao_atendimento_original": "_origem_hora_finalizacao_atendimento",
    "tempo_espera_informado_original": "_origem_tempo_espera_informado",
    "tempo_atendimento_informado_original": "_origem_tempo_atendimento_informado",
}

colunas_silver = [
    "id_chamado", "id_cliente", "id_cliente_referencia",
    "chamado_encontrado", "cliente_comparavel", "cliente_divergente",
    "hora_abertura_chamado", "_origem_hora_abertura_chamado",
    "hora_inicio_atendimento", "_origem_hora_inicio_atendimento",
    "hora_finalizacao_atendimento", "_origem_hora_finalizacao_atendimento",
    "abertura_no_periodo", "abertura_fora_periodo",
    "inicio_antes_abertura", "finalizacao_antes_inicio", "finalizacao_antes_abertura",
    "ordem_temporal_invalida",
    "tempo_espera_segundos", "tempo_espera_calculado_segundos",
    "tempo_atendimento_segundos", "tempo_atendimento_calculado_segundos",
    "tempo_total_segundos", "tempo_total_calculado_segundos",
    "_origem_tempo_espera_informado", "tempo_espera_informado_segundos",
    "tempo_espera_informado_ausente", "tempo_espera_informado_invalido",
    "tempo_espera_informado_negativo", "tempo_espera_comparavel",
    "tempo_espera_divergente", "diferenca_tempo_espera_segundos",
    "_origem_tempo_atendimento_informado", "tempo_atendimento_informado_segundos",
    "tempo_atendimento_informado_ausente", "tempo_atendimento_informado_invalido",
    "tempo_atendimento_informado_negativo", "tempo_atendimento_comparavel",
    "tempo_atendimento_divergente", "diferenca_tempo_atendimento_segundos",
    "_ingerido_em",
]

df_bronze = (
    spark.table(f"{BRONZE_SCHEMA}.tb_chamados_hora")
    .select(
        F.expr("try_cast(id_chamado as bigint)").alias("id_chamado"),
        "id_cliente",
        "hora_abertura_chamado",
        "hora_inicio_atendimento",
        "hora_finalizacao_atendimento",
        F.col("_ingerido_em"),
    )
)
linhas_bronze = df_bronze.count()

df_referencia = (
    spark.table(f"{BRONZE_SCHEMA}.tb_chamados")
    .select(
        F.expr("try_cast(id_chamado as bigint)").alias("id_chamado"),
        F.col("id_cliente").alias("id_cliente_referencia"),
        F.col("tempo_espera").alias("tempo_espera_informado_original"),
        F.col("tempo_atendimento").alias("tempo_atendimento_informado_original"),
    )
    .dropDuplicates()
)

verificar("id_chamado com registros conflitantes na referência de chamados",
          df_referencia.groupBy("id_chamado").count().filter(F.col("count") > 1).count(),
          critico=True)


def construir_silver_chamados_hora(bronze, referencia):
    base = bronze.dropDuplicates(COLUNAS_CHAMADOS_HORA)
    base = converter_horarios(base)
    base = calcular_tempos_horarios(base)
    base = sinalizar_periodo_horarios(base)
    base = cruzar_referencia_horarios(base, referencia)
    base = comparar_tempos_informados(base)
    for antes, depois in RENAMES_ORIGEM.items():
        base = base.withColumnRenamed(antes, depois)
    return base.select(*colunas_silver)


df_sem_dup = df_bronze.dropDuplicates(COLUNAS_CHAMADOS_HORA)
linhas_dedup = df_sem_dup.count()
print(f"chamados_hora: bronze={linhas_bronze} | sem duplicatas={linhas_dedup} | removidas={linhas_bronze - linhas_dedup}")

conflitos = (
    df_sem_dup.groupBy("id_chamado").count()
    .filter(F.col("count") > 1).count()
)
verificar("id_chamado com registros conflitantes após dedup", conflitos, critico=True)

df_silver = construir_silver_chamados_hora(df_bronze, df_referencia)

verificar("linhas perdidas no join com chamados",
          linhas_dedup - df_silver.count(), critico=True)

# Reconciliação do período
metricas_periodo = df_silver.agg(
    F.count("*").alias("registros"),
    F.sum(F.when(F.col("abertura_no_periodo"), 1).otherwise(0)).alias("no_periodo"),
    F.sum(F.when(F.col("abertura_fora_periodo"), 1).otherwise(0)).alias("fora_periodo"),
    F.sum(F.when(F.col("hora_abertura_chamado").isNull(), 1).otherwise(0)).alias("indisponivel"),
).first()

print(f"Período: total={metricas_periodo['registros']} | no_periodo={metricas_periodo['no_periodo']} | "
      f"fora_periodo={metricas_periodo['fora_periodo']} | indisponivel={metricas_periodo['indisponivel']}")

verificar("período: total != (no + fora + indisponível)",
          metricas_periodo["registros"]
          - (metricas_periodo["no_periodo"] + metricas_periodo["fora_periodo"] + metricas_periodo["indisponivel"]),
          critico=True)

soma_divergente = df_silver.filter(
    F.col("tempo_espera_segundos").isNotNull()
    & F.col("tempo_atendimento_segundos").isNotNull()
    & (F.col("tempo_total_segundos").isNull()
       | (F.col("tempo_total_segundos")
          != F.col("tempo_espera_segundos") + F.col("tempo_atendimento_segundos")))
).count()

verificar("tempos com soma divergente (espera + atend != total)",
          soma_divergente, critico=True)

verificar("id_chamado nulo",
          df_silver.filter(F.col("id_chamado").isNull()).count(), critico=True)

verificar("id_chamado duplicado",
          df_silver.count() - df_silver.select("id_chamado").distinct().count(), critico=True)

verificar("_ingerido_em nulo",
          df_silver.filter(F.col("_ingerido_em").isNull()).count(), critico=True)

verificar("clientes divergentes entre chamados_hora e chamados",
          df_silver.filter(F.col("cliente_divergente")).count())
verificar("clientes não comparáveis",
          df_silver.filter(~F.col("cliente_comparavel")).count())
verificar("tempos de espera divergentes (informado vs calculado)",
          df_silver.filter(F.col("tempo_espera_divergente")).count())
verificar("tempos de atendimento divergentes (informado vs calculado)",
          df_silver.filter(F.col("tempo_atendimento_divergente")).count())

# ---- Idempotência ----
df_reexecucao = construir_silver_chamados_hora(df_bronze, df_referencia)

if df_silver.schema != df_reexecucao.schema:
    raise Exception("Reexecução alterou o schema.")

verificar("reexecução: linhas ausentes",
          df_silver.exceptAll(df_reexecucao).count(), critico=True)
verificar("reexecução: linhas adicionais",
          df_reexecucao.exceptAll(df_silver).count(), critico=True)

# ---- Gravação ----
gravar_silver(df_silver, "tb_chamados_hora")
display(spark.table(f"{SILVER_SCHEMA}.tb_chamados_hora").limit(20))

✅ id_chamado com registros conflitantes na referência de chamados: 0
chamados_hora: bronze=131838 | sem duplicatas=131838 | removidas=0
✅ id_chamado com registros conflitantes após dedup: 0
✅ linhas perdidas no join com chamados: 0
Período: total=131838 | no_periodo=131640 | fora_periodo=198 | indisponivel=0
✅ período: total != (no + fora + indisponível): 0
✅ tempos com soma divergente (espera + atend != total): 0
✅ id_chamado nulo: 0
✅ id_chamado duplicado: 0
✅ _ingerido_em nulo: 0
⚠️ clientes divergentes entre chamados_hora e chamados: 1330
✅ clientes não comparáveis: 0
⚠️ tempos de espera divergentes (informado vs calculado): 6180
⚠️ tempos de atendimento divergentes (informado vs calculado): 27987
✅ reexecução: linhas ausentes: 0
✅ reexecução: linhas adicionais: 0
✅ vcredit_medallion.silver.tb_chamados_hora: 131838 linhas


id_chamado,id_cliente,id_cliente_referencia,chamado_encontrado,cliente_comparavel,cliente_divergente,hora_abertura_chamado,_origem_hora_abertura_chamado,hora_inicio_atendimento,_origem_hora_inicio_atendimento,hora_finalizacao_atendimento,_origem_hora_finalizacao_atendimento,abertura_no_periodo,abertura_fora_periodo,inicio_antes_abertura,finalizacao_antes_inicio,finalizacao_antes_abertura,ordem_temporal_invalida,tempo_espera_segundos,tempo_espera_calculado_segundos,tempo_atendimento_segundos,tempo_atendimento_calculado_segundos,tempo_total_segundos,tempo_total_calculado_segundos,_origem_tempo_espera_informado,tempo_espera_informado_segundos,tempo_espera_informado_ausente,tempo_espera_informado_invalido,tempo_espera_informado_negativo,tempo_espera_comparavel,tempo_espera_divergente,diferenca_tempo_espera_segundos,_origem_tempo_atendimento_informado,tempo_atendimento_informado_segundos,tempo_atendimento_informado_ausente,tempo_atendimento_informado_invalido,tempo_atendimento_informado_negativo,tempo_atendimento_comparavel,tempo_atendimento_divergente,diferenca_tempo_atendimento_segundos,_ingerido_em
80538,90787372340,90787372340,true,true,false,2026-02-09T13:10:23.000,09/02/2026 às 13:10:23,2026-02-09T13:10:23.000,09/02/2026 às 13:10:23,2026-02-09T13:13:18.000,09/02/2026 às 13:13:18,true,false,false,false,false,false,0,0,175,175,175,175,NULL,null,true,false,false,false,null,null,175,175,false,false,false,true,false,0,2026-10-26T20:21:33.924Z
100742,50041093143,50041093143,true,true,false,2026-04-30T18:36:34.000,30/04/2026 às 18:36:34,2026-04-30T18:36:34.000,30/04/2026 às 18:36:34,2026-04-30T18:38:52.000,30/04/2026 às 18:38:52,true,false,false,false,false,false,0,0,138,138,138,138,NULL,null,true,false,false,false,null,null,138,138,false,false,false,true,false,0,2026-10-26T20:21:33.924Z
36892,81455773123,81455773123,true,true,false,2025-08-12T07:27:29.000,12/08/2025 às 07:27:29,2025-08-12T07:27:29.000,12/08/2025 às 07:27:29,2025-08-12T07:29:44.000,12/08/2025 às 07:29:44,true,false,false,false,false,false,0,0,135,135,135,135,NULL,null,true,false,false,false,null,null,135,135,false,false,false,true,false,0,2026-10-26T20:21:33.924Z
106334,61700017855,61700017855,true,true,false,2026-05-07T09:13:26.000,07/05/2026 às 09:13:26,2026-05-07T09:13:26.000,07/05/2026 às 09:13:26,2026-05-07T09:14:51.000,07/05/2026 às 09:14:51,true,false,false,false,false,false,0,0,85,85,85,85,NULL,null,true,false,false,false,null,null,85,85,false,false,false,true,false,0,2026-10-26T20:21:33.924Z
50123,18337003488,18337003488,true,true,false,2025-10-23T15:20:47.000,23/10/2025 às 15:20:47,2025-10-23T15:20:47.000,23/10/2025 às 15:20:47,2025-10-23T15:21:05.000,23/10/2025 às 15:21:05,true,false,false,false,false,false,0,0,18,18,18,18,NULL,null,true,false,false,false,null,null,18,18,false,false,false,true,false,0,2026-10-26T20:21:33.924Z
107889,81982096597,81982096597,true,true,false,2026-05-11T19:13:46.000,11/05/2026 às 19:13:46,2026-05-11T19:13:46.000,11/05/2026 às 19:13:46,2026-05-11T19:16:51.000,11/05/2026 às 19:16:51,true,false,false,false,false,false,0,0,185,185,185,185,NULL,null,true,false,false,false,null,null,185,185,false,false,false,true,false,0,2026-10-26T20:21:33.924Z
47082,83975998622,83975998622,true,true,false,2025-09-29T10:29:59.000,29/09/2025 às 10:29:59,2025-09-29T10:30:17.000,29/09/2025 às 10:30:17,2025-09-29T10:52:50.000,29/09/2025 às 10:52:50,true,false,false,false,false,false,18,18,1353,1353,1371,1371,NULL,null,true,false,false,false,null,null,1,1,false,false,false,true,true,1352,2026-10-26T20:21:33.924Z
73467,45956633349,45956633349,true,true,false,2026-01-05T15:20:48.000,05/01/2026 às 15:20:48,2026-01-05T15:20:48.000,05/01/2026 às 15:20:48,2026-01-05T15:23:58.000,05/01/2026 às 15:23:58,true,false,false,false,false,false,0,0,190,190,190,190,NULL,null,true,false,false,false,null,null,190,190,false,false,false,true,false,0,2026-10-26T20:21:33.924Z
71125,58081297071,58081297071,true,true,false,2025-12-10T19:51:37.000,10/12/2025 às 19:51:37,2025-1

## Validação cruzada — 8 tabelas Silver

Verificações de integridade entre as tabelas publicadas:
1. Cada tabela existe, tem a contagem esperada e tem `_ingerido_em` sem nulos.
2. Integridade referencial (FK) — todos os ids nas FKs existem na dimensão correspondente.
3. Reconciliação do funil: chamados sem custo/pesquisa/horário correspondente.

**Contagens esperadas** (baseadas na execução atual; ajustar se a Bronze mudar):
- `tb_base_motivos`: 14
- `tb_canais`: 4
- `tb_clientes`: 47.705
- `tb_base_atendentes`: 25
- `tb_chamados`: 131.838 (após dedup)
- `tb_custos`: 130.288 (após dedup)
- `tb_pesquisa_satisfacao`: 81.021
- `tb_chamados_hora`: 131.838

In [0]:
TABELAS_ESPERADAS = {
    "tb_base_motivos":      14,
    "tb_canais":            4,
    "tb_clientes":          47705,
    "tb_base_atendentes":   25,
    "tb_chamados":          131838,
    "tb_custos":            130288,
    "tb_pesquisa_satisfacao": 81021,
    "tb_chamados_hora":     131838,
}

print("=" * 60)
print("1) Existência, contagem e _ingerido_em")
print("=" * 60)
for tabela, esperado in TABELAS_ESPERADAS.items():
    nome = f"{SILVER_SCHEMA}.{tabela}"
    try:
        df = spark.table(nome)
    except Exception as e:
        print(f"{FAIL} {tabela}: tabela não encontrada ({e})")
        continue

    n = df.count()
    tem_meta = "_ingerido_em" in df.columns
    meta_nula = (
        df.filter(F.col("_ingerido_em").isNull()).count() if tem_meta else -1
    )

    status = OK if (n == esperado and tem_meta and meta_nula == 0) else WARN
    print(f"{status} {tabela}: {n} linhas (esperado {esperado}) | "
          f"_ingerido_em: {'ok' if tem_meta and meta_nula == 0 else 'AUSENTE ou NULO'}")

print()
print("=" * 60)
print("2) Integridade referencial (FK)")
print("=" * 60)

chamados = spark.table(f"{SILVER_SCHEMA}.tb_chamados")
motivos = spark.table(f"{SILVER_SCHEMA}.tb_base_motivos")
canais = spark.table(f"{SILVER_SCHEMA}.tb_canais")
atendentes = spark.table(f"{SILVER_SCHEMA}.tb_base_atendentes")
clientes = spark.table(f"{SILVER_SCHEMA}.tb_clientes")
custos = spark.table(f"{SILVER_SCHEMA}.tb_custos")
pesquisas = spark.table(f"{SILVER_SCHEMA}.tb_pesquisa_satisfacao")
chamados_hora = spark.table(f"{SILVER_SCHEMA}.tb_chamados_hora")

# FK: chamados.id_motivo -> motivos.id_motivo
orfaos = chamados.join(
    motivos.select("id_motivo"), on="id_motivo", how="left_anti"
).count()
verificar("chamados.id_motivo sem correspondência em motivos", orfaos, critico=True)

# FK: chamados.id_canal -> canais.id_canal
orfaos = chamados.join(
    canais.select("id_canal"), on="id_canal", how="left_anti"
).count()
verificar("chamados.id_canal sem correspondência em canais", orfaos, critico=True)

# FK: chamados.id_atendente -> atendentes.id_atendente (nullable)
orfaos = (
    chamados.filter(F.col("id_atendente").isNotNull())
    .join(atendentes.select("id_atendente"), on="id_atendente", how="left_anti")
    .count()
)
verificar("chamados.id_atendente sem correspondência em atendentes", orfaos, critico=True)

# FK: chamados.id_cliente -> clientes.id_cliente (nullable, sinalizado por cliente_encontrado)
orfaos = (
    chamados.filter(F.col("cliente_encontrado"))
    .join(clientes.select("id_cliente"), on="id_cliente", how="left_anti")
    .count()
)
verificar("chamados.id_cliente (cliente_encontrado=true) sem correspondência", orfaos, critico=True)

# FK: custos.id_chamado -> chamados.id_chamado
orfaos = custos.join(
    chamados.select("id_chamado"), on="id_chamado", how="left_anti"
).count()
verificar("custos.id_chamado sem correspondência em chamados", orfaos, critico=True)

# FK: pesquisas.id_chamado -> chamados.id_chamado
orfaos = pesquisas.join(
    chamados.select("id_chamado"), on="id_chamado", how="left_anti"
).count()
verificar("pesquisa.id_chamado sem correspondência em chamados", orfaos, critico=True)

# FK: chamados_hora.id_chamado -> chamados.id_chamado
orfaos = chamados_hora.join(
    chamados.select("id_chamado"), on="id_chamado", how="left_anti"
).count()
verificar("chamados_hora.id_chamado sem correspondência em chamados", orfaos, critico=True)

print()
print("=" * 60)
print("3) Cobertura do funil (chamados com / sem enriquecimento)")
print("=" * 60)

metricas = chamados.agg(
    F.count("*").alias("total_chamados"),
    F.sum(F.when(F.col("cliente_encontrado"), 1).otherwise(0)).alias("com_cliente"),
    F.sum(F.when(F.col("atendente_identificado"), 1).otherwise(0)).alias("com_atendente"),
).first()

total = metricas["total_chamados"]
print(f"Chamados: {total}")
print(f"  com cliente encontrado:     {metricas['com_cliente']} ({100*metricas['com_cliente']/total:.2f}%)")
print(f"  com atendente identificado: {metricas['com_atendente']} ({100*metricas['com_atendente']/total:.2f}%)")

n_com_custo = chamados.join(
    custos.select("id_chamado").distinct(), on="id_chamado", how="left_semi"
).count()
n_com_pesquisa = chamados.join(
    pesquisas.select("id_chamado").distinct(), on="id_chamado", how="left_semi"
).count()
n_com_horario = chamados.join(
    chamados_hora.select("id_chamado").distinct(), on="id_chamado", how="left_semi"
).count()

print(f"  com custo registrado:       {n_com_custo} ({100*n_com_custo/total:.2f}%)")
print(f"  com pesquisa registrada:    {n_com_pesquisa} ({100*n_com_pesquisa/total:.2f}%)")
print(f"  com horário registrado:     {n_com_horario} ({100*n_com_horario/total:.2f}%)")

print()
print(f"{OK} validação cruzada concluída")

1) Existência, contagem e _ingerido_em
✅ tb_base_motivos: 14 linhas (esperado 14) | _ingerido_em: ok
✅ tb_canais: 4 linhas (esperado 4) | _ingerido_em: ok
✅ tb_clientes: 47705 linhas (esperado 47705) | _ingerido_em: ok
✅ tb_base_atendentes: 25 linhas (esperado 25) | _ingerido_em: ok
✅ tb_chamados: 131838 linhas (esperado 131838) | _ingerido_em: ok
✅ tb_custos: 130288 linhas (esperado 130288) | _ingerido_em: ok
✅ tb_pesquisa_satisfacao: 81021 linhas (esperado 81021) | _ingerido_em: ok
✅ tb_chamados_hora: 131838 linhas (esperado 131838) | _ingerido_em: ok

2) Integridade referencial (FK)
✅ chamados.id_motivo sem correspondência em motivos: 0
✅ chamados.id_canal sem correspondência em canais: 0
✅ chamados.id_atendente sem correspondência em atendentes: 0
✅ chamados.id_cliente (cliente_encontrado=true) sem correspondência: 0
✅ custos.id_chamado sem correspondência em chamados: 0
✅ pesquisa.id_chamado sem correspondência em chamados: 0
✅ chamados_hora.id_chamado sem correspondência em chama